# pandas 2/3 : Conditions, masques et indexation (`loc` / `iloc`)

**Objectifs** : filtrer les lignes d'un DataFrame selon des conditions (masques booléens), puis sélectionner précisément des lignes et colonnes avec `loc` (par étiquettes) et `iloc` (par positions).

*Durée estimée : 45 min.*

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/titanic.csv", index_col="PassengerId")
df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Ici, on a utilisé `PassengerId` comme **index** : les étiquettes des lignes vont de 1 à 891, alors que leurs positions vont de 0 à 890. Cette différence est le cœur de la distinction `loc` / `iloc`.

## 1. Conditions et masques

C'est exactement le même principe qu'avec NumPy : une comparaison produit un **masque** de booléens, que l'on utilise pour filtrer.

In [2]:
masque = df["Age"] < 18
print(masque.head())
print("Nombre de mineurs :", masque.sum())

PassengerId
1    False
2    False
3    False
4    False
5    False
Name: Age, dtype: bool
Nombre de mineurs : 113


In [3]:
mineurs = df[masque]          # ou directement : df[df["Age"] < 18]
mineurs.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C
11,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.0,1,1,PP 9549,16.7000,G6,S
15,0,3,"Vestrom, Miss. Hulda Amanda Adolfina",female,14.0,0,0,350406,7.8542,NaN,S
17,0,3,"Rice, Master. Eugene",male,2.0,4,1,382652,29.1250,NaN,Q


### Combiner des conditions

Comme avec NumPy : `&` (et), `|` (ou), `~` (non), avec **des parenthèses** autour de chaque condition.

In [4]:
# Femmes de 1ère classe
femmes_1 = df[(df["Sex"] == "female") & (df["Pclass"] == 1)]
print(len(femmes_1))

# Enfants ou personnes de plus de 70 ans
extremes = df[(df["Age"] < 10) | (df["Age"] > 70)]
print(len(extremes))

# Tous sauf les hommes
print(len(df[~(df["Sex"] == "male")]))

94
67
314


### Outils pratiques pour construire des masques

| Méthode | Rôle |
|---|---|
| `isin([...])` | la valeur appartient à une liste |
| `between(a, b)` | valeur comprise entre a et b (inclus) |
| `isna()` / `notna()` | valeur manquante / présente |
| `str.contains("txt")` | une chaîne contient un motif |

In [5]:
print(len(df[df["Embarked"].isin(["C", "Q"])]))
print(len(df[df["Age"].between(20, 30)]))
print(len(df[df["Age"].isna()]))
print(len(df[df["Name"].str.contains("Dr.", regex=False)]))

245
245
177
7


⚠️ Un `NaN` n'est **jamais** vrai pour une comparaison : `df["Age"] < 18` vaut `False` pour les âges inconnus. Ces passagers n'apparaîtront donc ni dans `Age < 18`, ni dans `Age >= 18`.

In [6]:
print((df["Age"] < 18).sum() + (df["Age"] >= 18).sum(), "sur", len(df))

714 sur 891


### Calculer sur un sous-ensemble

On enchaîne : filtrer, choisir une colonne, agréger.

In [7]:
print("Survie des femmes :", df[df["Sex"] == "female"]["Survived"].mean())
print("Survie des hommes :", df[df["Sex"] == "male"]["Survived"].mean())

Survie des femmes : 0.7420382165605095
Survie des hommes : 0.18890814558058924


---
### ✏️ Exercices (masques)

### ✏️ Exercice 1

Combien de passagers de 3ème classe ont survécu ?

In [1]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
print(len(df[(df["Pclass"] == 3) & (df["Survived"] == 1)]))
```

</details>

### ✏️ Exercice 2

Quel est le tarif moyen payé par les passagers qui ont embarqué à Cherbourg (`C`) ?

In [2]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
print(df[df["Embarked"] == "C"]["Fare"].mean())
```

</details>

### ✏️ Exercice 3

Quelle proportion des enfants de moins de 12 ans a survécu ? Comparez avec les adultes (12 ans et plus).

In [3]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
print(df[df["Age"] < 12]["Survived"].mean())
print(df[df["Age"] >= 12]["Survived"].mean())
```

</details>

## 2. `loc` : sélection par **étiquettes**

Syntaxe : `df.loc[lignes, colonnes]`. Les bornes d'un slice `loc` sont **incluses**.

In [20]:
print(df.loc[1])                       # la ligne dont l'index vaut 1 (une Series)

Survived                          0
Pclass                            3
Name        Braund, Mr. Owen Harris
Sex                            male
Age                            22.0
SibSp                             1
Parch                             0
Ticket                    A/5 21171
Fare                           7.25
Cabin                           NaN
Embarked                          S
Name: 1, dtype: object


In [21]:
df.loc[1:3]                             # lignes d'index 1, 2 ET 3

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [22]:
print(df.loc[1:3, "Name"])                      # une colonne -> Series
df.loc[1:3, ["Name", "Age"]]                      # plusieurs colonnes -> DataFrame

PassengerId
1                              Braund, Mr. Owen Harris
2    Cumings, Mrs. John Bradley (Florence Briggs Th...
3                               Heikkinen, Miss. Laina
Name: Name, dtype: str


,Name,Age
PassengerId,,
1,"Braund, Mr. Owen Harris",22.0
2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0
3,"Heikkinen, Miss. Laina",26.0


In [23]:
df.loc[1:3, "Name":"Age"]                    # slice de colonnes (bornes incluses aussi)

,Name,Sex,Age
PassengerId,,,
1,"Braund, Mr. Owen Harris",male,22.0
2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0
3,"Heikkinen, Miss. Laina",female,26.0


## 3. `iloc` : sélection par **positions**

Syntaxe : `df.iloc[lignes, colonnes]`, avec des entiers comme pour NumPy (0 = première ligne). Les bornes de fin sont **exclues**.

In [24]:
print(df.iloc[0])          # la première ligne, quel que soit son index
df.iloc[0:3]               # positions 0, 1 et 2 (3 exclu)

Survived                          0
Pclass                            3
Name        Braund, Mr. Owen Harris
Sex                            male
Age                            22.0
SibSp                             1
Parch                             0
Ticket                    A/5 21171
Fare                           7.25
Cabin                           NaN
Embarked                          S
Name: 1, dtype: object


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [25]:
df.iloc[:3, [2, 3, 4]]      # 3 premières lignes, colonnes en positions 2, 3 et 4


,Name,Sex,Age
PassengerId,,,
1,"Braund, Mr. Owen Harris",male,22.0
2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0
3,"Heikkinen, Miss. Laina",female,26.0


In [26]:
df.iloc[-3:]               # 3 dernières lignes

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


### Comparaison directe

Sur nos données, l'étiquette `10` correspond à la position `9`. Observez la différence :

In [27]:
print(df.loc[10:12, "Name"])    # étiquettes 10, 11, 12
print()
print(df.iloc[10:12]["Name"])    # positions 10 et 11 -> étiquettes 11 et 12

PassengerId
10    Nasser, Mrs. Nicholas (Adele Achem)
11        Sandstrom, Miss. Marguerite Rut
12               Bonnell, Miss. Elizabeth
Name: Name, dtype: str

PassengerId
11    Sandstrom, Miss. Marguerite Rut
12           Bonnell, Miss. Elizabeth
Name: Name, dtype: str


| | `loc` | `iloc` |
|---|---|---|
| se base sur | **étiquettes** | **positions** |
| bornes d'un slice | incluses | fin exclue |
| accepte un masque booléen | oui | pas directement (utiliser `.values`) |

## 4. `loc` avec un masque

C'est l'usage le plus fréquent : choisir **les lignes par une condition** et **les colonnes par leur nom**.

In [28]:
df.loc[df["Age"] < 5, ["Name", "Age", "Survived"]]

,Name,Age,Survived
PassengerId,,,
8,"Palsson, Master. Gosta Leonard",2.00,0
11,"Sandstrom, Miss. Marguerite Rut",4.00,1
17,"Rice, Master. Eugene",2.00,0
44,"Laroche, Miss. Simonne Marie Anne Andree",3.00,1
64,"Skoog, Master. Harald",4.00,0
79,"Caldwell, Master. Alden Gates",0.83,1
120,"Andersson, Miss. Ellis Anna Maria",2.00,0
165,"Panula, Master. Eino Viljami",1.00,0
172,"Rice, Master. Arthur",4.00,0


### Modifier des valeurs

Pour **modifier** un sous-ensemble, on utilise `loc` sur le DataFrame d'origine. Ne tentez pas `df[df["Age"] < 5]["Survived"] = ...` : cela travaille sur une copie temporaire et ne modifie pas `df`.

In [29]:
df2 = df.copy()

# Remplacer les âges manquants par la médiane
mediane = df2["Age"].median()
df2.loc[df2["Age"].isna(), "Age"] = mediane
print(df2["Age"].isna().sum())

# Créer une colonne catégorielle à partir d'une condition
df2["Mineur"] = False
df2.loc[df2["Age"] < 18, "Mineur"] = True
df2["Mineur"].value_counts()

0


Mineur
False    778
True     113
Name: count, dtype: int64

## 5. Changer l'index : `set_index` / `reset_index`

Un index porteur de sens facilite l'accès avec `loc`.

In [30]:
par_nom = df.set_index("Name")
par_nom.loc["Braund, Mr. Owen Harris", ["Age", "Fare"]]

Age     22.00
Fare     7.25
Name: Braund, Mr. Owen Harris, dtype: float64

In [31]:
par_nom.reset_index().head(3)    # on remet un index 0, 1, 2... et Name redevient colonne

,Name,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,"Braund, Mr. Owen Harris",0,3,male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,1,female,38.0,1,0,PC 17599,71.2833,C85,C
2,"Heikkinen, Miss. Laina",1,3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## Récapitulatif

| Je veux… | Code |
|---|---|
| filtrer les lignes | `df[condition]` |
| combiner des conditions | `(c1) & (c2)`, `(c1) \| (c2)`, `~(c)` |
| appartenance, intervalle, manquant | `isin`, `between`, `isna` |
| par étiquettes | `df.loc[lignes, colonnes]` |
| par positions | `df.iloc[lignes, colonnes]` |
| filtrer + choisir des colonnes | `df.loc[condition, ["c1", "c2"]]` |
| modifier une sélection | `df.loc[condition, "col"] = valeur` |

---
## Exercices (`loc` / `iloc`)

### ✏️ Exercice 4

Avec `iloc`, affichez le nom et l'âge des 5 derniers passagers du fichier. *Indice : `Name` est en position 2 et `Age` en position 4.*

In [4]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
df.iloc[-5:, [2, 4]]
```

</details>

### ✏️ Exercice 5

Avec `loc`, affichez les noms et classes (`Pclass`) des passagers d'index 100 à 105 (inclus).

In [5]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
df.loc[100:105, ["Name", "Pclass"]]
```

</details>

### ✏️ Exercice 6

Avec `loc` et un masque, affichez nom, âge et tarif des passagers de 1ère classe ayant payé plus de 200.

In [6]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
df.loc[(df["Pclass"] == 1) & (df["Fare"] > 200), ["Name", "Age", "Fare"]]
```

</details>

### ✏️ Exercice 7

Dans une copie de `df`, remplacez les valeurs manquantes de `Embarked` par `"S"` (le port le plus fréquent). Vérifiez qu'il ne reste plus de manquants dans cette colonne.

In [7]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
d = df.copy()
d.loc[d["Embarked"].isna(), "Embarked"] = "S"
print(d["Embarked"].isna().sum())
```

</details>

### ✏️ Exercice 8

**Défi** : parmi les passagers dont le nom contient `"Mrs."`, quelle proportion a survécu ? Comparez avec ceux dont le nom contient `"Miss."`.

In [8]:
# Votre code ici

<details>
<summary>💡 Voir la solution</summary>

```python
print(df[df["Name"].str.contains("Mrs.", regex=False)]["Survived"].mean())
print(df[df["Name"].str.contains("Miss.", regex=False)]["Survived"].mean())
```

</details>